# FF Alpha、DGTW 与机制检验结果整理 Notebook

本 notebook 用来整理并解释以下四个资产定价扩展结果文件：

1. `ff_alpha_results.csv`
2. `dgtw_results_summary.csv`
3. `mechanism_interaction_regressions.csv`
4. `mechanism_triple_sort_tech_size_ai.csv`

这些文件位于：`asset_pricing_outputs_v2/`

本 notebook 的目标不是重新从原始 CRSP/Compustat 数据估计所有模型，而是把已经生成的结果表系统化展示、解释，并形成论文中 **Asset Pricing / Risk Adjustment / Mechanism** 部分可以直接引用的结果框架。


## 0. 结果阅读口径

本 notebook 中所有收益率系数均以 **decimal return** 表示。例如：

- `-0.0925` = `-9.25%`
- `-0.0785` = `-7.85%`
- `0.0203` = `2.03%`

显著性星号含义：

| 星号 | 含义 |
|---|---|
| `***` | 1% 水平显著 |
| `**` | 5% 水平显著 |
| `*` | 10% 水平显著 |
| 无星号 | 不显著 |

核心解释：如果 AI portfolio spread 或 AI coefficient 为负，表示 **AI-narrative firms 在未来收益上跑输 non-AI firms**。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

# Run from the submission root or code/asset_pricing_notebooks directory.
BASE = Path.cwd()
if BASE.name == 'asset_pricing_notebooks':
    BASE = BASE.parent.parent
elif BASE.name == 'code':
    BASE = BASE.parent
OUT = BASE / 'results' / 'asset_pricing' / 'asset_pricing_outputs_v2'

OUT = BASE / "asset_pricing_outputs_v2"

paths = {
    "ff_alpha": OUT / "ff_alpha_results.csv",
    "dgtw": OUT / "dgtw_results_summary.csv",
    "mechanism_interaction": OUT / "mechanism_interaction_regressions.csv",
    "triple_sort": OUT / "mechanism_triple_sort_tech_size_ai.csv",
}

for name, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

ff_alpha = pd.read_csv(paths["ff_alpha"])
dgtw = pd.read_csv(paths["dgtw"])
interaction = pd.read_csv(paths["mechanism_interaction"])
triple = pd.read_csv(paths["triple_sort"])

print("Files loaded from:", OUT)
for name, df in [("ff_alpha", ff_alpha), ("dgtw", dgtw), ("interaction", interaction), ("triple", triple)]:
    print(f"{name}: {df.shape}")


In [ ]:
def pct(x, digits=2):
    """Format decimal return as percentage string."""
    if pd.isna(x):
        return ""
    return f"{100*x:.{digits}f}%"

def add_pct_columns(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c + "_pct"] = out[c].map(lambda x: pct(x))
    return out

print("Loaded result files preview:")
display(ff_alpha.head())
display(dgtw.head())
display(interaction)
display(triple.head())


# 1. Fama-French Factor Alpha Tests

## 1.1 检验目的

Fama-French alpha test 用来回答：

> AI-narrative portfolio 的负收益，是否只是因为它暴露在已知风险因子上？

具体做法是把 AI portfolio 的 long-short spread 作为因变量，回归到标准资产定价因子上：

\[
R^{AI-LS}_t = \alpha + \beta_1(MKT-RF)_t + \beta_2 SMB_t + \beta_3 HML_t + \beta_4 RMW_t + \beta_5 CMA_t + \beta_6 MOM_t + \epsilon_t
\]

如果 `alpha` 仍显著为负，说明 AI spread 在标准因子调整后依然存在。更谨慎的表述是：

> negative spread survives standard factor adjustments, and is difficult to reconcile with standard risk-based explanations.

注意：本项目季度样本较短，`FF5+MOM` 在 4Q outcome 下只有约 8 个季度观测，因此 FF alpha 结果应作为辅助证据；更稳健的主风险调整结果应看 DGTW。


In [ ]:
# 只保留主组合：AI dummy portfolio。
ff_ai = ff_alpha[ff_alpha["portfolio"].str.contains("AI dummy", case=False, na=False)].copy()
ff_ai = ff_ai.sort_values(["outcome", "portfolio", "model"])

cols = ["portfolio", "outcome", "model", "n_obs", "alpha", "alpha_sig", "alpha_t", "alpha_p", "r2"]
ff_ai_display = ff_ai[cols].copy()
ff_ai_display["alpha_pct"] = ff_ai_display["alpha"].map(pct)

print("Fama-French alpha tests: AI dummy portfolios")
display(ff_ai_display[["portfolio", "outcome", "model", "n_obs", "alpha_pct", "alpha_sig", "alpha_t", "alpha_p", "r2"]])


In [ ]:
# 重点表：4Q outcome, AI dummy portfolios。
ff_4q = ff_ai[ff_ai["outcome"].eq("ret_future_4q")].copy()
ff_4q_display = ff_4q[["portfolio", "model", "n_obs", "alpha", "alpha_sig", "alpha_t", "r2"]].copy()
ff_4q_display["alpha_pct"] = ff_4q_display["alpha"].map(pct)

print("核心结果：4Q FF alpha")
display(ff_4q_display[["portfolio", "model", "n_obs", "alpha_pct", "alpha_sig", "alpha_t", "r2"]])


In [ ]:
# 图：4Q alpha by factor model。
plot_df = ff_4q.copy()
model_order = ["CAPM", "FF3", "Carhart4", "FF5+MOM"]
plot_df["model"] = pd.Categorical(plot_df["model"], categories=model_order, ordered=True)
plot_df = plot_df.sort_values(["portfolio", "model"])

fig, ax = plt.subplots(figsize=(9, 4.5))
for portfolio, g in plot_df.groupby("portfolio"):
    ax.plot(g["model"].astype(str), g["alpha"] * 100, marker="o", label=portfolio)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("4Q Fama-French Alpha: AI Dummy Portfolios")
ax.set_ylabel("Alpha (%)")
ax.set_xlabel("Factor model")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.show()


## 1.2 主要解读

核心结果集中在 `ret_future_4q`：

- `AI dummy EW` 在 `FF5+MOM` 模型下 alpha 约为 **-9.25%**，显著为负。
- `AI dummy VW` 在 `FF5+MOM` 模型下 alpha 约为 **-5.73%**，显著为负。

解释：即使扣除了 `MKT_RF`、`SMB`、`HML`、`RMW`、`CMA` 和 `MOM`，AI-narrative firms 相对于 non-AI firms 的未来 4Q underperformance 仍然存在。

谨慎表述：由于季度时间序列较短，FF alpha 不应作为唯一主证据，而应与 DGTW 和 panel regression 一起构成 converging evidence。


# 2. DGTW Characteristic-Adjusted Abnormal Returns

## 2.1 检验目的

DGTW 方法用 firm-level characteristic-matched benchmark 调整收益。直观来说：

> 对每家公司，先找到 size、BM、momentum 等特征相近的 benchmark portfolio，再计算公司实际收益减去 benchmark 收益。这样得到的 abnormal return 更像是在同类公司内部比较。

相比 FF alpha，DGTW 的优势是：

- 保留 firm-quarter 层面的横截面信息；
- 不需要用很短的季度时间序列估计 6 个因子载荷；
- 在本项目这种 2020–2022 短样本中更稳健。

因此，DGTW 是本项目中最重要的 risk-adjusted robustness evidence。


In [ ]:
dgtw_display = dgtw.copy()
dgtw_display["diff_pct"] = dgtw_display["diff"].map(pct)
dgtw_display["ai_abret_pct"] = pd.to_numeric(dgtw_display["ai_abret"], errors="coerce").map(pct)
dgtw_display["non_abret_pct"] = pd.to_numeric(dgtw_display["non_abret"], errors="coerce").map(pct)

cols = ["spec", "outcome", "test", "n_obs_ai", "n_obs_non", "ai_abret_pct", "non_abret_pct", "diff_pct", "sig", "t", "p"]
print("DGTW characteristic-adjusted abnormal returns")
display(dgtw_display[cols])


In [ ]:
# 重点表：4Q DGTW 结果。
dgtw_4q = dgtw_display[dgtw_display["outcome"].eq("ret_future_4q")].copy()
print("核心结果：4Q DGTW abnormal return, AI minus non-AI")
display(dgtw_4q[["spec", "test", "diff_pct", "sig", "t", "p"]])


In [ ]:
# 图：4Q DGTW diff。
plot_df = dgtw_4q.copy()
labels = plot_df["spec"] + "\n" + plot_df["test"].str.replace("quarterly TS mean ", "TS ", regex=False)
values = plot_df["diff"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(values)), values, color="#4C78A8")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("4Q DGTW Characteristic-Adjusted Spread: AI minus Non-AI")
ax.set_ylabel("DGTW abnormal return spread (%)")
ax.set_xticks(range(len(values)))
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.grid(axis="y", alpha=0.3)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.1f}%", ha="center", va="top" if v < 0 else "bottom", fontsize=9)
plt.tight_layout()
plt.show()


## 2.2 主要解读

DGTW 的 4Q 结果非常稳定：

- `Size 5x BM 5`：AI minus non-AI abnormal return 约 **-7.85%**（quarterly TS HAC）
- `Size 5x BM 5 (VW)`：约 **-8.15%**（quarterly TS HAC）
- `Size 3x BM 3x Mom 3`：约 **-6.07%**（quarterly TS HAC）

三种 characteristic adjustment 都显示，AI-narrative firms 在未来 4Q 显著跑输 matched benchmark。

这说明负向收益并不是简单由 size、book-to-market 或 momentum 这些已知特征驱动。


# 3. Interaction Regressions: AI × Tech 与 AI × Size

## 3.1 检验目的

Interaction regression 用来检验 AI narrative 的负向收益预测是否会随着公司信息环境变化而变弱或增强。

核心交互项：

- `AI × Tech`：科技行业是否会削弱 AI narrative 的负向收益预测？
- `AI × Size`：公司越大，AI narrative 的负向收益预测是否越弱？

如果交互项为正，说明在该维度下 AI 的负向效应被缓和。


In [ ]:
inter_display = interaction.copy()
for c in ["AI_main", "AI_x_inter"]:
    inter_display[c + "_pct"] = inter_display[c].map(pct)

cols = [
    "outcome", "interaction_with", "n_obs",
    "AI_main_pct", "AI_main_sig", "AI_main_t", "AI_main_p",
    "AI_x_inter_pct", "AI_x_inter_sig", "AI_x_inter_t", "AI_x_inter_p"
]
print("Interaction regressions")
display(inter_display[cols])


## 3.2 主要解读

`AI_main` 表示基准组中 AI narrative 对未来收益的预测系数；`AI_x_inter` 表示 Tech 或 Size 是否会改变这一系数。

需要谨慎阅读：

- `AI_main` 在 1Q 和 4Q 中均为负，说明 AI narrative 与未来收益负相关。
- `AI × Tech` 和 `AI × Size` 的交互项方向为正，表示科技行业和大公司倾向于削弱 AI 的负向收益预测。
- 但部分交互项统计显著性较弱，因此这一结果应作为机制提示，而非强因果证据。

保守解释：结果与“信息环境越好，narrative mispricing 越弱”的解释一致，但不能直接证明具体行为机制。


# 4. Tech × Size Triple Sort

## 4.1 检验目的

单独比较 Tech vs Non-Tech 不够，因为科技属性和公司规模会相互作用。

Tech × Size triple sort 将样本分为：

- `Tech` vs `Non-Tech`
- `Small` / `Mid` / `Large`
- 在每个子样本内部计算 `AI firms - non-AI firms` 的未来收益 spread

这一步用于回答：AI narrative 的负向收益预测到底集中在哪些公司？


In [ ]:
triple_display = triple.copy()
triple_display["spread_pct"] = triple_display["spread"].map(pct)

print("Tech × Size triple sort: full table")
display(triple_display[["outcome", "tech", "size_tercile", "weight", "n_quarters", "spread_pct", "sig", "hac_t", "hac_p"]])


In [ ]:
# 重点表：4Q equal-weighted triple sort。
triple_4q_ew = triple_display[
    triple_display["outcome"].eq("ret_future_4q") & triple_display["weight"].eq("ew")
].copy()

print("核心结果：4Q EW Tech × Size triple sort")
display(triple_4q_ew[["tech", "size_tercile", "n_quarters", "spread_pct", "sig", "hac_t", "hac_p"]])

pivot = triple_4q_ew.pivot(index="tech", columns="size_tercile", values="spread")
pivot = pivot[["Small", "Mid", "Large"]]
print("Pivot table: spread in decimals")
display(pivot)


In [ ]:
# 热力图：4Q EW spread。
fig, ax = plt.subplots(figsize=(7, 3.5))
values = pivot.values * 100
im = ax.imshow(values, cmap="RdBu", vmin=-16, vmax=16)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title("4Q EW Spread: AI minus Non-AI by Tech × Size")

for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        ax.text(j, i, f"{values[i, j]:.1f}%", ha="center", va="center", color="black", fontsize=11)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Spread (%)")
plt.tight_layout()
plt.show()


## 4.2 主要解读

4Q equal-weighted triple sort 是机制部分最关键的表。

主要模式：

| 子样本 | 4Q spread | 解读 |
|---|---:|---|
| `Tech × Large` | 约 -3.1%，不显著 | 大型科技公司讲 AI 已经被市场充分预期，负向收益预测弱 |
| `Tech × Small` | 约 -14.3%，显著 | 小型科技公司讲 AI 更像 unexpected narrative，未来收益跑输明显 |
| `Non-Tech × Mid` | 约 -15.1%，显著 | 中型非科技公司讲 AI 转型，市场更可能过度外推 |
| `Non-Tech × Large` | 约 -6.9%，显著 | 大型非科技公司仍存在负向收益预测，但弱于 Non-Tech Mid |

因此，更准确的机制叙事不是简单的“non-tech mispricing 最强”，而是：

> AI narrative 的负向收益预测集中在投资者较难预期 AI narrative 的地方，尤其是 small technology firms 和 mid-sized non-technology firms；large technology firms 中负向预测较弱或不显著。


# 5. 汇总结论

本 notebook 对四个扩展结果文件的核心结论如下：

1. **FF alpha tests**：`AI dummy` portfolio 的 4Q alpha 在 `FF5+MOM` 调整后仍显著为负，说明负向 spread survives standard factor adjustments。但由于季度样本短，FF alpha 应作为辅助证据。

2. **DGTW characteristic-adjusted returns**：AI-narrative firms 在 4Q 内相对于 matched benchmark 跑输约 **6%–8%**，是本项目最稳健的 risk-adjusted evidence。

3. **Interaction regressions**：`AI × Tech` 与 `AI × Size` 的方向显示 Tech 和 Size 会缓和 AI 的负向收益预测，与信息环境解释一致，但应谨慎表述为 suggestive evidence。

4. **Tech × Size triple sort**：负向收益预测集中在 `Tech × Small` 和 `Non-Tech × Mid`，而 `Tech × Large` 中较弱或不显著。这支持更精确的故事：AI narrative mispricing is concentrated where AI narratives are less expected by investors.

论文写作中建议把 DGTW 放在主稳健性位置，把 FF alpha 放在补充 risk-adjustment 位置，把 Tech × Size triple sort 作为机制部分核心表。


In [ ]:
# 可选：保存整理后的核心表，方便论文写作引用。
summary_dir = OUT / "notebook_summary_tables"
summary_dir.mkdir(exist_ok=True)

ff_4q_display.to_csv(summary_dir / "summary_ff_alpha_4q.csv", index=False)
dgtw_4q[["spec", "outcome", "test", "diff", "sig", "t", "p"]].to_csv(summary_dir / "summary_dgtw_4q.csv", index=False)
inter_display.to_csv(summary_dir / "summary_interaction_regressions.csv", index=False)
triple_4q_ew.to_csv(summary_dir / "summary_triple_sort_4q_ew.csv", index=False)

print("Saved summary tables to:", summary_dir)
